# Checking a Model

A model that runs and produces a smooth profile is not yet a correct model. This
tutorial shows the checks that separate the two, using tools that `PyMRM`
provides:

- `describe_bc` writes a boundary-condition dictionary out as the equation it imposes;
- `pymrm.checks.residual_check` judges a Newton solution;
- `pymrm.checks.check_jacobian` compares a Jacobian with finite differences;
- `pymrm.checks.observed_orders` runs a grid-refinement study;
- `pymrm.checks.find_roots` finds every steady state of a scalar balance.

The same checks apply to any model, whether you wrote it yourself or a coding
agent wrote it for you.

## The model

Steady diffusion with a second-order reaction in a catalyst slab of
half-thickness $L$, with symmetry at the centre and a fixed surface
concentration:

$$D \frac{d^2 c}{dx^2} = k\, c^2, \qquad \frac{dc}{dx}(0) = 0, \qquad c(L) = c_s.$$

The quantity of interest is the effectiveness factor: the actual consumption
rate divided by the rate at surface conditions throughout the slab. At steady
state all reactant enters through the surface, so

$$\eta = \frac{D\, \frac{dc}{dx}(L)}{k\, c_s^2\, L}.$$

There is no closed-form solution for second-order kinetics, which makes this a
good case for checks that do not need one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from pymrm import (construct_grad, construct_div, compute_boundary_values,
                   describe_bc, NumJac, newton)
from pymrm.checks import check_jacobian, residual_check, observed_orders, find_roots

L = 1.0    # slab half-thickness
D = 1.0    # effective diffusivity
k = 20.0   # second-order rate constant
c_s = 1.0  # surface concentration


class SlabModel:
    # Steady diffusion with second-order reaction in a slab, n cells.

    def __init__(self, n):
        self.x_f = np.linspace(0.0, L, n + 1)
        self.x_c = 0.5 * (self.x_f[:-1] + self.x_f[1:])
        # x = 0: dc/dx = 0 (symmetry); x = L: c = c_s
        self.bc = ({"a": 1.0, "b": 0.0, "d": 0.0}, {"a": 0.0, "b": 1.0, "d": c_s})
        shape = (n, 1)  # one field on n cells: keep the trailing field axis
        grad_mat, grad_bc = construct_grad(shape, self.x_f, self.x_c, self.bc)
        div_mat = construct_div(shape, self.x_f, nu=0)
        self.jac_diff = div_mat @ (-D * grad_mat)
        self.g_bc = div_mat @ (-D * grad_bc)
        self.numjac = NumJac(shape)
        self.c = np.full(shape, c_s)

    def reaction(self, c):
        return -k * c**2

    def residual(self, c):
        r, jac_r = self.numjac(self.reaction, c)
        g = self.jac_diff @ c.reshape((-1, 1)) + self.g_bc - r.reshape((-1, 1))
        return g, self.jac_diff - jac_r

    def solve(self):
        result = newton(self.residual, self.c)
        self.c = result.x.reshape(self.c.shape)
        return result

    def effectiveness(self):
        # the gradient AT the surface face, from the boundary condition
        _, _, _, dcdx_surface = compute_boundary_values(self.c, self.x_f, self.x_c, self.bc)
        return float(D * np.ravel(dcdx_surface)[0] / (k * c_s**2 * L))

## Check 1: the boundary conditions say what you mean

A boundary dictionary `{"a", "b", "d"}` imposes $a\, \partial c/\partial n + b\, c = d$
with $n$ the **outward** normal. At the lower end the outward normal points in the
$-x$ direction, so a flux condition written there in terms of $dc/dx$ changes
sign. `describe_bc` prints what each dictionary imposes, in terms of $x$:

In [ ]:
model = SlabModel(50)
print(describe_bc(model.bc, model.x_f))

Compare each line with the physical condition: zero gradient at $x = 0$ and
$c = c_s$ at $x = L$.

## Check 2: did Newton really converge?

`newton` reports success when its last step is smaller than an **absolute**
tolerance. That is not the same as a small residual: with very small or very
large unknowns the step can be small while the equations are not satisfied.
`residual_check` judges the solution by a scale-free backward error instead.

In [ ]:
result = model.solve()
print(f"newton: success = {result.success}, iterations = {result.nit}")
print("residual check:", residual_check(model.residual, model.c))
print(f"effectiveness factor, 50 cells: {model.effectiveness():.6f}")

## Check 3: is the Jacobian right?

A wrong Jacobian does not always make Newton fail; it may converge slowly, or to
the right answer from a good initial guess and to nothing from a poor one.
`check_jacobian` compares the Jacobian returned by the residual function with
central differences. A check is only useful if it can fail, so we also feed it
a deliberately broken residual function, with the sign of the reaction
Jacobian flipped:

In [ ]:
print("correct Jacobian:", check_jacobian(model.residual, model.c)["ok"])


def broken_residual(c):
    g, _ = model.residual(c)
    _, jac_r = model.numjac(model.reaction, c)
    return g, model.jac_diff + jac_r  # wrong sign on the reaction part


print("broken Jacobian: ", check_jacobian(broken_residual, model.c))

## Check 4: grid refinement

Refine the grid by a constant factor and look at how the answer changes.
`observed_orders` computes the order of convergence from each three successive
values, so it needs no exact solution. The finite-volume scheme used here is
second order, so the observed order should approach 2. The study also gives a
Richardson-extrapolated value and an estimate of the error on the finest grid.

In [ ]:
def effectiveness(n):
    m = SlabModel(n)
    m.solve()
    return m.effectiveness()


study = observed_orders(effectiveness, (25, 50, 100, 200))
for n, value in zip((25, 50, 100, 200), study["values"]):
    print(f"n = {n:4d}: eta = {value:.8f}")
print("observed orders:", np.round(study["orders"], 2))
print(f"extrapolated: {study['extrapolated']:.8f}, error estimate at n = 200: {study['error_estimate']:.1e}")

## Check 5: an independent route

Agreement between grids shows that the discretisation converges, not that it
converges to the solution of the right equations. Solve the same problem once
more by a different method, here SciPy's `solve_bvp` on the second-order ODE
written as a first-order system:

In [ ]:
def ode(x, y):
    return np.vstack([y[1], k / D * y[0]**2])


def boundary(ya, yb):
    return np.array([ya[1], yb[0] - c_s])


x_init = np.linspace(0.0, L, 50)
bvp = solve_bvp(ode, boundary, x_init, np.vstack([np.ones(50), np.zeros(50)]), tol=1e-10, max_nodes=100000)
eta_bvp = D * bvp.sol(L)[1] / (k * c_s**2 * L)
print(f"solve_bvp: eta = {eta_bvp:.8f} (status {bvp.status})")
print(f"difference with the extrapolated finite-volume value: {abs(eta_bvp - study['extrapolated']):.1e}")

plt.plot(model.x_c, model.c[:, 0], "o", ms=4, label="pymrm, 50 cells")
plt.plot(x_init, bvp.sol(x_init)[0], "-", label="solve_bvp")
plt.xlabel("x")
plt.ylabel("c")
plt.title("Second-order reaction in a slab")
plt.legend()
plt.show()

## Read boundary values from the face

The operators work with values and gradients at the cell **faces**. A flux
estimated from the last cell centre instead is half a cell short of the
boundary. On the 50-cell grid that costs a few percent, far more than
the discretisation error found above:

In [ ]:
dx_half = model.x_f[-1] - model.x_c[-1]
eta_cell = D * (c_s - model.c[-1, 0]) / dx_half / (k * c_s**2 * L)
print(f"from compute_boundary_values: {model.effectiveness():.6f}")
print(f"from the last cell centre:    {eta_cell:.6f}")

## Check 6: find every steady state

A nonlinear model can have several steady states, and Newton finds the one
nearest its initial guess. When a question reduces to a scalar balance, scan it
for every sign change instead. As an example, an adiabatic CSTR with a
first-order exothermic reaction, in dimensionless form: conversion $X$,
temperature $\theta = 1 + \beta X$ and Damkohler number
$Da(\theta) = Da_0 \exp(\gamma (1 - 1/\theta))$, with steady states where

$$f(X) = X - \frac{Da(\theta)}{1 + Da(\theta)} = 0.$$

In [ ]:
gamma, beta = 20.0, 0.3


def cstr_balance(X, Da_0):
    theta = 1.0 + beta * X
    Da = Da_0 * np.exp(gamma * (1.0 - 1.0 / theta))
    return X - Da / (1.0 + Da)


for Da_0 in (0.05, 0.08, 0.10):
    found = find_roots(lambda X: cstr_balance(X, Da_0), 0.0, 1.0)
    print(f"Da_0 = {Da_0:.2f}: X = {np.round(found['roots'], 4)} ({found['message']})")

At $Da_0 = 0.08$ there are three steady states: a low-conversion, a
high-conversion and an unstable middle one. A single Newton solve would have
reported one of them.

## Summary

| Check | Tool | Catches |
|---|---|---|
| boundary conditions | `describe_bc` | sign errors from the outward normal |
| convergence of Newton | `pymrm.checks.residual_check` | a small step without a small residual |
| Jacobian | `pymrm.checks.check_jacobian` | wrong derivatives, wrong signs |
| grid refinement | `pymrm.checks.observed_orders` | unresolved gradients, bugs that spoil the order |
| independent route | a different method, a closed form or a limit | solving the wrong equations consistently |
| face values | `compute_boundary_values` | fluxes read half a cell from the boundary |
| multiple steady states | `pymrm.checks.find_roots` | reporting one solution of several |

## Exercise

- Increase $k$ to 500 and repeat the refinement study. At which grid size does
  the observed order approach 2? Try a grid refined towards the surface with
  `non_uniform_grid`.
- Break the model on purpose, one change at a time, and see which checks
  notice: add the reaction term in `residual` with the wrong sign but keep the
  Jacobian; or replace $c^2$ by $c^3$ in `reaction` (the residual and its
  Jacobian stay consistent, so only the independent route can tell).
- For the CSTR, find the range of $Da_0$ with three steady states.